# JS/CSS Asset Builder (`dist/`)

> Extracts `#| export_js`, `#| export_js_iife`, `#| export_css`, and `#| export_ft` cells from feature notebooks in `nbs/craft/` into a standalone `dist/` build directory, copying any not-yet-migrated root assets so `dist/` is always complete and verifiable via `--assets-dir dist`.

In [ ]:
#| default_exp build

In [ ]:
#| export
import json, shutil, subprocess
from collections import defaultdict
from fastcore.nbio import read_nb
from fastcore.utils import *
from fastcore.xml import FT, ft, to_xml
import fastcore.xml as fx

from craft_dev.auth import REPO_ROOT

In [ ]:
#| export
DEFAULT_DIST_DIR = REPO_ROOT/'dist'
DEFAULT_CRAFT_NBS_DIR = REPO_ROOT/'nbs'/'craft'
CRAFT_ROOT_ASSETS = ('CRAFT.css', 'CRAFT.js')
CRAFT_SUB_ASSETS = ('craft_md.js', 'hotkey_manager.js', 'hotkey_builtins.js', 'quercus-treeview.css', 'quercus-treeview.js')
CRAFT_ASSET_FILES = (*CRAFT_ROOT_ASSETS, *CRAFT_SUB_ASSETS)
_EXPORT_RE = re.compile(r'^#\|\s*export[_-](js[_-]iife|js|css|ft)\s+(\S+)(?:\s+(\S+))?\s*$')
_PREFIX_RE = re.compile(r'^(\d+)_')

## Extracting JS & CSS Cells from Notebooks

Feature notebooks mark exported asset cells with `#| export_js <filename>`, `#| export_js_iife <filename> [group]`, `#| export_css <filename>`, or `#| export_ft <filename>` (paired with `#| eval: false` for raw JS/CSS so `nbdev-test` does not execute them as Python).

Use `#| export_js_iife <filename> <group>` to split a large IIFE across multiple narrated notebook cells while preserving a single shared lexical scope in the built JS.

In [ ]:
#| export
def svg_icon(paths, size=15, sw=1.6, **kw):
    "Build an inline SVG `FT` icon from a path string or list of `FT`/path strings."
    kids = [ft('path', d=p) if isinstance(p, str) else p for p in listify(paths)]
    return ft('svg', *kids, aria_hidden='true', viewBox='0 0 24 24', width=str(size), height=str(size), fill='none', stroke='currentColor',
        stroke_width=str(sw), stroke_linecap='round', stroke_linejoin='round', **kw)

def _cls(*parts): return ' '.join(p.strip() for p in parts if p and p.strip())

def craft_overlay(*kids, cls='', **kw):
    "Modal backdrop overlay `FT` component (`craft-overlay`)."
    return fx.Div(*kids, klass=_cls('craft-overlay', cls), **kw)

def craft_card(*kids, cls='', **kw):
    "Elevated dialog/popover card `FT` component (`uk-card craft-card`)."
    return fx.Div(*kids, klass=_cls('uk-card craft-card', cls), **kw)

def craft_menu(*kids, cls='', **kw):
    "Floating popover/context menu `FT` component (`craft-menu`)."
    return fx.Div(*kids, klass=_cls('craft-menu', cls), **kw)

def craft_btn(*kids, variant='default', size='sm', cls='', **kw):
    "FrankenUI-styled button `FT` component (`uk-btn uk-btn-<variant>`)."
    v = f'uk-btn-{variant}' if variant else ''
    s = f'uk-btn-{size}' if size else ''
    kw.setdefault('type', 'button')
    return fx.Button(*kids, klass=_cls('uk-btn', v, s, cls), **kw)

def craft_input(cls='', **kw):
    "FrankenUI-styled input `FT` component (`uk-input`)."
    kw.setdefault('type', 'text')
    return fx.Input(klass=_cls('uk-input', cls), **kw)

In [ ]:
#| export
def ft_templates_js(tpls, var='__craftTpl'):
    "Compile `{name: FT | str}` templates into a compact JS `window[var]` assignment."
    rendered = {k: to_xml(v, indent=False).strip() if isinstance(v, (FT, tuple, list)) else str(v) for k, v in tpls.items()}
    return f"window.{var} = Object.assign(window.{var} || {{}}, {json.dumps(rendered, ensure_ascii=False)});"

def ft_config_js(cfg, var='__craftCfg'):
    "Serialize Python config/tokens dict into a compact JS `window[var]` assignment."
    return f"window.{var} = Object.assign(window.{var} || {{}}, {json.dumps(cfg, ensure_ascii=False)});"

In [ ]:
#| export
def _parse_asset_cell(src):
    lines = src.splitlines()
    kind, target, group, body = None, None, None, []
    for ln in lines:
        if m := _EXPORT_RE.match(ln): kind, target, group = m.group(1).replace('-', '_'), m.group(2), m.group(3)
        elif ln.startswith('#|'): continue
        else: body.append(ln)
    return kind, target, group, '\n'.join(body).strip()

def _eval_ft_cell(code, ns):
    ns.pop('TEMPLATES', None)
    ns.pop('CONFIG', None)
    exec(code, ns)
    parts = []
    if tpls := ns.get('TEMPLATES'): parts.append(ft_templates_js(tpls))
    if cfg := ns.get('CONFIG'): parts.append(ft_config_js(cfg))
    return '\n'.join(parts)

def _wrap_iife(group, parts):
    body = '\n\n'.join(f'/* #{cid} */\n{code}' if cid else code for cid, code in parts)
    return f"(() => {{ /* iife:{group} */\n{body}\n}})();"

def extract_nb_assets(nb_path):
    "Extract `{target_file: [(cell_id, code), ...]}` from `#| export_js`, `#| export_js_iife`, `#| export_css`, and `#| export_ft` cells."
    nb = read_nb(nb_path)
    out, iifes, ft_ns = defaultdict(list), {}, None
    for c in nb.cells:
        if c.cell_type != 'code': continue
        kind, target, group, code = _parse_asset_cell(c.source)
        if not (target and code): continue
        cid = c.get('id', '')
        if kind == 'js_iife':
            key = (target, group or 'default')
            if key not in iifes:
                slot = [cid, []]
                iifes[key] = slot
                out[target].append(slot)
            iifes[key][1].append((cid, code))
            continue
        if kind == 'ft':
            if ft_ns is None:
                ft_ns = {k: getattr(fx, k) for k in dir(fx) if not k.startswith('_')}
                ft_ns.update(L=L, merge=merge, listify=listify, svg_icon=svg_icon, craft_overlay=craft_overlay,
                    craft_card=craft_card, craft_menu=craft_menu, craft_btn=craft_btn, craft_input=craft_input,
                    ft_templates_js=ft_templates_js, ft_config_js=ft_config_js)
            code = _eval_ft_cell(code, ft_ns)
        if code: out[target].append((cid, code))
    res = {}
    for target, items in out.items():
        grp_by_slot = {id(slot): grp for (tgt, grp), slot in iifes.items() if tgt == target}
        res[target] = [(it[0], _wrap_iife(grp_by_slot[id(it)], it[1])) if isinstance(it, list) else it for it in items]
    return res

In [ ]:
import tempfile
from fastcore.nbio import mk_cell, new_nb, write_nb
from fastcore.test import test_eq, test_fail

`extract_nb_assets` collects `#| export_js` and `#| export_css` cells in cell order and strips `#|` directive lines.

In [ ]:
with tempfile.TemporaryDirectory() as td:
    nb_p = Path(td)/'demo.ipynb'
    nb = new_nb([
        mk_cell('#| export_js demo.js\n#| eval: false\nconst a = 1;'),
        mk_cell('#| export_css demo.css\n#| eval: false\n.x { color: red; }'),
        mk_cell('x = 1 + 1'),
    ])
    write_nb(nb, nb_p)
    got = extract_nb_assets(nb_p)
    test_eq(sorted(got), ['demo.css', 'demo.js'])
    test_eq(got['demo.js'][0][1], 'const a = 1;')
    test_eq(got['demo.css'][0][1], '.x { color: red; }')

`#| export_ft` evaluates FastHTML / `fastcore.xml` `FT` trees at build time and emits compact HTML/SVG strings into `window.__craftTpl`, ready for `htmx.process(...)` in the browser.

In [ ]:
with tempfile.TemporaryDirectory() as td:
    nb_p = Path(td)/'ft_demo.ipynb'
    nb = new_nb([
        mk_cell('#| export_ft demo.js\nTEMPLATES = dict(btn=Button(svg_icon("M12 6v12"), "Add", type="button", hx_post="/add_below_", hx_include="#kid", hx_swap="none"))'),
    ])
    write_nb(nb, nb_p)
    js = extract_nb_assets(nb_p)['demo.js'][0][1]
    assert 'hx-post=\\"/add_below_\\"' in js and 'viewbox=\\"0 0 24 24\\"' in js and '__craftTpl' in js

`#| export_js_iife <target> <group>` wraps all cells in a notebook that share the same `(target, group)` into one `(() => { ... })();` block at the position of the group's first cell, so a large JS module can be split across narrated cells without losing shared lexical scope.

In [ ]:
with tempfile.TemporaryDirectory() as td:
    nb_p = Path(td)/'iife_demo.ipynb'
    nb = new_nb([
        mk_cell('#| export_js_iife demo.js counter\n#| eval: false\nlet count = 0;'),
        mk_cell('Explanation between sections', cell_type='markdown'),
        mk_cell('#| export_js_iife demo.js counter\n#| eval: false\nwindow.inc = () => ++count;'),
    ])
    write_nb(nb, nb_p)
    items = extract_nb_assets(nb_p)['demo.js']
    test_eq(len(items), 1)
    js = items[0][1]
    assert js.startswith('(() => { /* iife:counter */') and 'let count = 0;' in js and 'window.inc = () => ++count;' in js and js.endswith('})();')

## Building `dist/`

`build_dist` merges all feature notebooks in `nbs/craft/` (in sorted filename order), copies any remaining assets from the repo root so `dist/` is always a complete drop-in `--assets-dir`, and syntax-checks every generated `.js` file with `node --check`.

In [ ]:
#| export
def _check_js_syntax(js_path):
    res = subprocess.run(['node', '--check', str(js_path)], capture_output=True, text=True)
    if res.returncode: raise RuntimeError(f"node --check failed for {js_path}:\n{res.stderr}")

def _check_unique_prefixes(nb_paths, nbs_dir):
    by_pfx = defaultdict(list)
    for p in nb_paths:
        if m := _PREFIX_RE.match(p.name): by_pfx[m.group(1)].append(p.name)
    dups = {k: v for k, v in by_pfx.items() if len(v) > 1}
    if dups:
        msg = '; '.join(f"{k}_: {', '.join(v)}" for k, v in sorted(dups.items()))
        raise ValueError(f"Duplicate numeric notebook prefixes in {nbs_dir}: {msg}")

def build_dist(
    out_dir=DEFAULT_DIST_DIR, # Output directory for assembled CRAFT assets
    nbs_dir=DEFAULT_CRAFT_NBS_DIR, # Directory of feature notebooks containing `#| export_*` cells
    src_dir=REPO_ROOT, # Repo root used to copy not-yet-migrated assets and `vscode-icons/`
    check_js=True, # Run `node --check` on every `.js` file in `out_dir`
    check_prefixes=True, # Fail if two notebooks in `nbs_dir` share the same numeric prefix
):
    "Assemble a complete CRAFT asset bundle in `out_dir` from feature notebooks plus unmigrated root files."
    out_dir, nbs_dir, src_dir = Path(out_dir), Path(nbs_dir), Path(src_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    chunks = defaultdict(list)
    if nbs_dir.is_dir():
        nb_paths = sorted(nbs_dir.ls(file_exts='.ipynb'))
        if check_prefixes: _check_unique_prefixes(nb_paths, nbs_dir)
        for nb_p in nb_paths:
            rel = nb_p.relative_to(src_dir) if nb_p.is_relative_to(src_dir) else nb_p.name
            for target, items in extract_nb_assets(nb_p).items():
                for cid, code in items:
                    chunks[target].append(f"/* %% {rel} #{cid} */\n{code}")

    report = {}
    for target, parts in sorted(chunks.items()):
        text = '\n\n'.join(parts) + '\n'
        dest = out_dir/target
        dest.mk_write(text)
        report[target] = {'source': 'nb', 'lines': len(text.splitlines())}

    for name in CRAFT_ASSET_FILES:
        if name in report: continue
        src = src_dir/name
        if src.is_file():
            dest = out_dir/name
            dest.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(src, dest)
            report[name] = {'source': 'copied', 'lines': len(src.read_text(encoding='utf-8').splitlines())}

    expected = {(out_dir/target).resolve() for target in report}
    for p in list(out_dir.rglob('*')):
        if not p.is_file(): continue
        if (out_dir/'vscode-icons') in p.parents: continue
        if p.resolve() not in expected: p.unlink()

    icons_src, icons_dst = src_dir/'vscode-icons', out_dir/'vscode-icons'
    if icons_dst.exists(): shutil.rmtree(icons_dst)
    if icons_src.is_dir(): shutil.copytree(icons_src, icons_dst)

    if check_js:
        for target in report:
            if target.endswith('.js'): _check_js_syntax(out_dir/target)
    return report

`build_dist` lets a single feature notebook override one asset while copying the rest into an isolated build directory.

In [ ]:
with tempfile.TemporaryDirectory() as td:
    td = Path(td)
    nb_dir, out_dir = td/'nbs', td/'dist'
    nb_dir.mkdir()
    write_nb(new_nb([
        mk_cell('#| export_js_iife hotkey_builtins.js demo\n#| eval: false\nconst _ok = 1;'),
        mk_cell('#| export_js_iife hotkey_builtins.js demo\n#| eval: false\nwindow.__demo = _ok;'),
    ]), nb_dir/'00_hk.ipynb')
    out_dir.mkdir()
    (out_dir/'stale.js').write_text('// stale')
    (out_dir/'vscode-icons').mkdir()
    (out_dir/'vscode-icons'/'stale.svg').write_text('<svg/>')
    rep = build_dist(out_dir=out_dir, nbs_dir=nb_dir, src_dir=REPO_ROOT, check_js=True)
    test_eq(rep['hotkey_builtins.js']['source'], 'nb')
    test_eq(rep['CRAFT.js']['source'], 'copied')
    test_eq((out_dir/'CRAFT.css').is_file(), True)
    test_eq((out_dir/'stale.js').exists(), False)
    test_eq((out_dir/'vscode-icons'/'stale.svg').exists(), False)

`build_dist` raises a `ValueError` if two feature notebooks share the same numeric prefix, preventing ambiguous JS/CSS load order.

In [ ]:
with tempfile.TemporaryDirectory() as td:
    td = Path(td)
    nb_dir, out_dir = td/'nbs', td/'dist'
    nb_dir.mkdir()
    write_nb(new_nb([mk_cell('#| export_js a.js\n#| eval: false\nconst a = 1;')]), nb_dir/'00_first.ipynb')
    write_nb(new_nb([mk_cell('#| export_js b.js\n#| eval: false\nconst b = 2;')]), nb_dir/'00_second.ipynb')
    test_fail(lambda: build_dist(out_dir=out_dir, nbs_dir=nb_dir, src_dir=REPO_ROOT), contains='Duplicate numeric notebook prefixes')

Verify the repository shell installers (`setup.sh`, `zsh-setup.sh`) pass `bash -n`.

In [ ]:
for sh in ('setup.sh', 'zsh-setup.sh'):
    res = subprocess.run(['bash', '-n', str(REPO_ROOT/sh)], capture_output=True, text=True)
    assert res.returncode == 0, f"bash -n failed for {sh}:\n{res.stderr}"

setup_txt = (REPO_ROOT/'setup.sh').read_text(encoding='utf-8')
m = re.search(r'^files=\(([^)]+)\)$', setup_txt, re.M)
assert m, "Could not find files=(...) in setup.sh"
sh_files = {t for t in m.group(1).split() if not t.startswith('vscode-icons/')}
test_eq(sh_files, set(CRAFT_ASSET_FILES))
m_root = re.search(r'^\s*([^\n)]+)\)\s*rel=\$f\s*;;$', setup_txt, re.M)
assert m_root, "Could not find root asset case in setup.sh"
test_eq(set(m_root.group(1).split('|')), set(CRAFT_ROOT_ASSETS))